# Experiment 7 - Transformer Models: GPT and BERT


In [ ]:
!pip -q install transformers torch

In [ ]:
import torch, pandas as pd
from transformers import pipeline
device = 0 if torch.cuda.is_available() else -1
print("Pipeline device:", "GPU" if device == 0 else "CPU")

Pipeline device: CPU


In [ ]:
# GPT-2 text generation: three prompts in a cricket coaching context.
generator = pipeline("text-generation", model="gpt2", device=device)
gpt_prompts = [
    "A beginner cricket batter can improve their balance by",
    "A coach reviewing a cover drive should first observe",
    "A useful 15-minute batting practice session includes"
]
gpt_outputs=[]
for prompt in gpt_prompts:
    out=generator(prompt, max_new_tokens=55, do_sample=True, temperature=0.8, top_k=50,
                  pad_token_id=generator.tokenizer.eos_token_id)[0]["generated_text"]
    gpt_outputs.append({"Prompt":prompt,"GPT-2 output":out})
    print("\nPROMPT:",prompt,"\nOUTPUT:",out)
gpt_df=pd.DataFrame(gpt_outputs)
display(gpt_df)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


PROMPT: A beginner cricket batter can improve their balance by 
OUTPUT: A beginner cricket batter can improve their balance by being faster and sharper than a beginner. This method is very similar to Kullen batsman who hits an inside-out swing.

Another method of playing a beginner cricket batter is to use a different batting style altogether. If you know how to play cricket, you

PROMPT: A coach reviewing a cover drive should first observe 
OUTPUT: A coach reviewing a cover drive should first observe the situation and make a decision as to what the best strategy for the player should be. A coach reviewing a cover drive should first observe the situation and make a decision as to what the best strategy for the player should be.

The best cover drive is a short one

PROMPT: A useful 15-minute batting practice session includes 
OUTPUT: A useful 15-minute batting practice session includes a variety of games from batting practice to batting practice. The sessions are also designed to give

,Prompt,GPT-2 output
0,A beginner cricket batter can improve their ba...,A beginner cricket batter can improve their ba...
1,A coach reviewing a cover drive should first o...,A coach reviewing a cover drive should first o...
2,A useful 15-minute batting practice session in...,A useful 15-minute batting practice session in...


In [ ]:
# Observe the effect of decoding parameters on the same prompt.
prompt="A cricket coach gives feedback by"
for temp in [0.3, 0.8, 1.2]:
    out=generator(prompt,max_new_tokens=45,do_sample=True,temperature=temp,top_k=50,
                  pad_token_id=generator.tokenizer.eos_token_id)[0]["generated_text"]
    print(f"\nTemperature={temp}\n{out}")


Temperature=0.3
A cricket coach gives feedback by phone to his players.

"I've been told that if you don't get the ball, you don't get the ball. I've been told that if you don't get the ball, you don't get

Temperature=0.8
A cricket coach gives feedback by himself on the outcome of a Test match. Photo: Reuters

A cricket coach gives feedback by himself on the outcome of a Test match. Photo: Reuters

A cricket coach gives feedback by himself on the outcome of

Temperature=1.2
A cricket coach gives feedback by signing and fielding. "You have to think that way of doing things," he added, adding that "all the things you do are important too."

In other words, you need to take steps now to create pressure


In [ ]:
# BERT masked-token prediction.
fill_mask = pipeline("fill-mask", model="bert-base-uncased", device=device)
masked = "A cricket batter should maintain [MASK] while playing a shot."
bert_mask_outputs = fill_mask(masked)
print("Masked sentence:", masked)
for item in bert_mask_outputs[:5]:
    print(item["sequence"], "| score:", round(item["score"],4))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Masked sentence: A cricket batter should maintain [MASK] while playing a shot.
a cricket batter should maintain balance while playing a shot. | score: 0.1222
a cricket batter should maintain control while playing a shot. | score: 0.0953
a cricket batter should maintain composure while playing a shot. | score: 0.0891
a cricket batter should maintain calm while playing a shot. | score: 0.0874
a cricket batter should maintain distance while playing a shot. | score: 0.0557


In [ ]:
# A second BERT-family understanding task: sentiment classification.
# This is a fine-tuned DistilBERT classifier, while BERT-base above performs masked-token prediction.
sentiment = pipeline("sentiment-analysis",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                     device=device)
sentences = [
    "The coaching session was helpful and encouraging.",
    "The instructions were confusing and frustrating."
]
for sentence in sentences:
    print(sentence, "=>", sentiment(sentence)[0])

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

The coaching session was helpful and encouraging. => {'label': 'POSITIVE', 'score': 0.9998587369918823}
The instructions were confusing and frustrating. => {'label': 'NEGATIVE', 'score': 0.9995136260986328}


In [ ]:
comparison = pd.DataFrame([
    ["Architecture","Decoder-only","Encoder-only (BERT); this notebook also uses a fine-tuned DistilBERT classifier"],
    ["Primary purpose","Autoregressive text generation","Language understanding / contextual representations"],
    ["Training objective","Next-token prediction","Masked-token prediction (pre-training)"],
    ["Input/output","Prompt → generated continuation","Masked sentence → token candidates; text → class label"],
    ["Strength","Flexible continuation and generation","Contextual understanding for supported tasks"],
    ["Limitation","May continue incoherently or invent claims","Not a general-purpose long-form text generator"]
],columns=["Parameter","GPT-2","BERT / BERT-family"])
display(comparison)

,Parameter,GPT-2,BERT / BERT-family
0,Architecture,Decoder-only,Encoder-only (BERT); this notebook also uses a...
1,Primary purpose,Autoregressive text generation,Language understanding / contextual representa...
2,Training objective,Next-token prediction,Masked-token prediction (pre-training)
3,Input/output,Prompt → generated continuation,Masked sentence → token candidates; text → cla...
4,Strength,Flexible continuation and generation,Contextual understanding for supported tasks
5,Limitation,May continue incoherently or invent claims,Not a general-purpose long-form text generator
